In [4]:
%load_ext autoreload
%autoreload 2
import pprint, json, math, os, sys, camelot
# sys.path.append(os.path.abspath(dir_path))
import subprocess

import fitz, pdfplumber, ocrmypdf, pprint
import pandas as pd
import numpy as np
from collections import defaultdict
from app.parse_table import TableParser
from app.utils import Helper, PDFTableExtractor

helper = Helper()

RANDOM SAMPLING FOR COLS

In [ ]:
import random
import fitz
helper = Helper()

def _extract_words(page,bbox):
    items = []
    words = page.get_text("words")
    for w in words:
        x0, y0, x1, y1, text = w[:5]

        text = text.strip()        
        if not text:
            continue

        if bbox:
            bx0, by0, bx1, by1 = bbox
            if not (x0 >= bx0 and y0 >= by0 and x1 <= bx1 and y1 <= by1):
                continue

        items.append({
            "x0": x0, "y0": y0,
            "x1": x1, "y1": y1,
            "text": text,
            "x_center": (x0 + x1) / 2,
            "y_center": (y0 + y1) / 2,
            "height": y1 - y0
        })
    return items


def col_sample(page,items, iteration = 60):

    # items = _extract_spans(page,bbox)
    if not items:
        return []

    page_h = page.rect.height
    y_hits = []

    for _ in range(iteration):  # stable than 50
        x = random.uniform(0, page_h)

        for item in items:
            if item["y0"] <= x <= item["y1"]:
                y_hits.append([item["x_center"],item["text"]])

    return y_hits

if __name__ == "__main__":
    
    path = r"JOHN.pdf"
    doc = fitz.open(path)
    
    page = doc[0]
    bbox = [72.87, 136.97, 546.11, 611.09]
    items = _extract_words(page,bbox)
    
    x_hits = col_sample(page,items,iteration=1000)
    
    import pprint
    
    # pprint.pprint(sorted(x_hits))
    # print(sorted(x_hits))

In [ ]:
from app.utils import Helper
import subprocess

helper = Helper()

pdf_path = "KOT.pdf"

masked_pdf = helper.mask_outside_bboxes(
    pdf_path,
    [(157.47, 88.49, 386.51, 852.41)]
)

subprocess.Popen([masked_pdf], shell=True)

# df = extract_table(
#     pdf_path=masked_pdf,   
#     page_no=1,
#     bbox=None,             
#     x_lines=[220, 415],   
#     # top_left=(430, 120),   
#     # top_right=(610, 120)   
# )

# print(df)

# save_to_excel(df, "soutput.xlsx")

HANDLER OF TABLE EXTRACTOR

In [ ]:
#new code
from app.utils import Helper
from table_fetch import FetchTable
jsn_path = r"annot.json"
helper = Helper()
configs = helper.load_json(jsn_path)


pdf_path = r"AVIVA.pdf"

df = FetchTable.new_handler(
    pdf_path,
    configs
)

df.to_csv("SAMPLEEEEEEEEE.csv", index=None)


[123.43758392333984, 138.17769622802734, 152.91768646240234, 167.65767669677734, 183.01766204833984, 199.22762298583984, 215.43758392333984, 231.67769622802734, 247.88765716552734, 264.0976257324219, 280.3077087402344, 296.5176696777344, 312.7276306152344, 328.9376525878906, 345.1476745605469, 361.3576354980469, 377.5676574707031, 393.8076477050781, 410.0176696777344, 426.2276306152344, 442.4376525878906, 458.6476745605469, 474.8576354980469, 491.0676574707031, 507.2776641845703, 523.4876098632812, 539.7276000976562, 555.9376220703125, 572.1476440429688, 588.3576049804688, 604.567626953125, 620.7776489257812, 636.9876098632812, 653.1976318359375, 669.4075927734375, 685.6176147460938, 701.8475952148438, 718.0576171875, 734.2676391601562]


AttributeError: 'dict' object has no attribute 'to_csv'

In [ ]:
#new code sheet wise
import pandas as pd
from app.utils import Helper
from table_fetch import FetchTable
from pathlib import Path
jsn_path = r"annot.json"
helper = Helper()
configs = helper.load_json(jsn_path)


pdf_path = r"ZEE1.pdf"

def determine_numeric_cols(df:pd.DataFrame, threshold = 0.6)->list:

    is_numeric_like = lambda col: (lambda s: sum(c in "0123456789,.- " for c in s) / len(s) if len(s) else 0)(" ".join(col.astype(str)))
    
    numeric_cols = []
    df_cols = df.columns
    # print(df_cols)
    for col in df_cols:
        ratio = is_numeric_like(df[col])
        if ratio>=threshold:
            numeric_cols.append(col)
    return numeric_cols

filter_numeric = lambda x: "".join(c for c in str(x) if c in "0123456789.,-")
encode_utf = lambda text: str(text).encode('latin1', errors='ignore').decode('utf-8', errors='ignore')



content = FetchTable.renew_handler(
    pdf_path,
    configs
)

file_name = Path(pdf_path).name

xls_path = file_name.replace(".pdf",".xlsx")

with pd.ExcelWriter(xls_path, engine="openpyxl") as writer:
    for page_n, df in content.items():
        sheet_name = f"Page_{page_n}"
        
        numeric_cols = determine_numeric_cols(df)
        other_cols = [i for i in df.columns if i not in numeric_cols]
        # print(numeric_cols)

        df1 = df.copy()
        for col in numeric_cols:
            df1[col] = df[col].apply(filter_numeric)
        
        for col in other_cols:
            df1[col] = df[col].apply(encode_utf)
        
        df1.to_excel(writer, sheet_name=sheet_name, index=False)


In [1]:
from table_fetch import FetchTable
import pandas as pd
from app.utils import Helper
pth = r"pdf_url.json"
helper = Helper()
configs = helper.load_json(pth)

cfg = configs["apollo"]
path = cfg["path"]

def determine_numeric_cols(df:pd.DataFrame, threshold = 0.6)->list:

    is_numeric_like = lambda col: (lambda s: sum(c in "0123456789,.- " for c in s) / len(s) if len(s) else 0)(" ".join(col.astype(str)))
    
    numeric_cols = []
    df_cols = df.columns
    # print(df_cols)
    for col in df_cols:
        ratio = is_numeric_like(df[col])
        if ratio>=threshold:
            numeric_cols.append(col)
    return numeric_cols

filter_numeric = lambda x: "".join(c for c in str(x) if c in "0123456789.,-")
encode_utf = lambda text: str(text).encode('latin1', errors='ignore').decode('utf-8', errors='ignore')




df = FetchTable.handler(path,cfg)
df.head(20)

# df.to_excel(path.replace(".pdf",".xlsx"), index=False)
numeric_cols = determine_numeric_cols(df.iloc[:,:-1])
other_cols = [i for i in df.columns if i not in numeric_cols]
print(numeric_cols)

df1 = df.copy()
for col in numeric_cols:
   df1[col] = df[col].apply(filter_numeric)
   
for col in other_cols:
   df1[col] = df[col].apply(encode_utf)
df1.to_excel(path.replace(".pdf",".xlsx"), index=False)

[1, 2, 3, 4, 5, 6]
